In [ ]:
# ── Imports and setup ──

import pandas as pd
import numpy as np
import os
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
OUT = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT, exist_ok=True)

In [ ]:
# ── Load delta CLR data ──

delta = pd.read_csv(os.path.join(OUT, "delta_otu_clr.csv"))

META_COLS = ['study', 'subject_id', 'treatment']
otu_cols  = [c for c in delta.columns if c not in META_COLS]

X      = delta[otu_cols].values
y      = (delta['treatment'] == 'fiber').astype(int).values
groups = delta['study'].values

print(f"delta shape   : {delta.shape}")
print(f"OTU features  : {len(otu_cols)}")
print(f"Subjects      : {len(y)}  (fiber={y.sum()}, control={(1-y).sum()})")
print(f"\nStudy breakdown:")
print(delta.groupby(['study','treatment']).size().to_string())

In [ ]:
# ── LOSO-CV function ──

def loso_cv(X, y, groups, model_type='rf'):
    studies = np.unique(groups)
    results = []

    rf_grid = {
        'n_estimators'    : [100, 300, 500],
        'max_features'    : ['sqrt', 'log2', 0.1],
        'min_samples_leaf': [1, 3]
    }
    xgb_grid = {
        'n_estimators' : [100, 300, 500],
        'max_depth'    : [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.3]
    }

    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for test_study in studies:
        train_idx = np.where(groups != test_study)[0]
        test_idx  = np.where(groups == test_study)[0]

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if len(np.unique(y_test)) < 2:
            print(f"  SKIP  {test_study:25s} — single class in test set (fiber-only study)")
            continue

        t0 = time.time()

        if model_type == 'rf':
            base = RandomForestClassifier(
                class_weight='balanced', random_state=42, n_jobs=-1
            )
            gs = GridSearchCV(base, rf_grid, cv=inner_cv,
                              scoring='roc_auc', n_jobs=-1)
        else:
            ratio = (y_train == 0).sum() / (y_train == 1).sum()
            base = xgb.XGBClassifier(
                eval_metric='logloss', random_state=42,
                scale_pos_weight=ratio, n_jobs=-1
            )
            gs = GridSearchCV(base, xgb_grid, cv=inner_cv,
                              scoring='roc_auc', n_jobs=-1)

        gs.fit(X_train, y_train)
        proba = gs.best_estimator_.predict_proba(X_test)[:, 1]
        auc   = roc_auc_score(y_test, proba)
        elapsed = time.time() - t0

        results.append({
            'study'      : test_study,
            'n_test'     : len(y_test),
            'n_fiber'    : int(y_test.sum()),
            'n_control'  : int((1 - y_test).sum()),
            'AUC'        : round(auc, 4),
            'best_params': gs.best_params_,
            'time_min'   : round(elapsed / 60, 1)
        })
        print(f"  DONE  {test_study:25s}  n={len(y_test):3d}  "
              f"AUC={auc:.4f}  {elapsed/60:.1f}min  {gs.best_params_}")

    return results

print("LOSO-CV function defined.")
print(f"\nExpected evaluable folds: Baxter, Dahl, Deehan, Healey, Morales (5 folds)")
print(f"Expected skipped folds  : Kovatcheva, Liu, Rasmussen, Tap, Venkataraman (fiber-only)")

In [ ]:
# ── Random Forest LOSO-CV ──

print("=== Random Forest LOSO-CV on Delta CLR ===\n")
t_start = time.time()

rf_results = loso_cv(X, y, groups, model_type='rf')

total = (time.time() - t_start) / 60
rf_df = pd.DataFrame(rf_results)

print(f"\nTotal time: {total:.1f} min")
print(f"\n{'study':<25} {'n_test':>6} {'n_fiber':>8} {'n_control':>10} {'AUC':>7}")
print("-" * 62)
for _, row in rf_df.iterrows():
    print(f"{row['study']:<25} {row['n_test']:>6} {row['n_fiber']:>8} {row['n_control']:>10} {row['AUC']:>7.4f}")

print("-" * 62)
print(f"\nMean AUC : {rf_df['AUC'].mean():.4f} ± {rf_df['AUC'].std():.4f}")

In [ ]:
# ── XGBoost LOSO-CV ──

print("=== XGBoost LOSO-CV on Delta CLR ===\n")
t_start = time.time()

xgb_results = loso_cv(X, y, groups, model_type='xgb')

total = (time.time() - t_start) / 60
xgb_df = pd.DataFrame(xgb_results)

print(f"\nTotal time: {total:.1f} min")
print(f"\n{'study':<25} {'n_test':>6} {'n_fiber':>8} {'n_control':>10} {'AUC':>7}")
print("-" * 62)
for _, row in xgb_df.iterrows():
    print(f"{row['study']:<25} {row['n_test']:>6} {row['n_fiber']:>8} {row['n_control']:>10} {row['AUC']:>7.4f}")

print("-" * 62)
print(f"\nMean AUC : {xgb_df['AUC'].mean():.4f} ± {xgb_df['AUC'].std():.4f}")

# Side by side comparison
print("\n=== RF vs XGBoost comparison ===")
print(f"\n{'study':<25} {'RF AUC':>8} {'XGB AUC':>9}")
print("-" * 45)
for _, row in rf_df.iterrows():
    xgb_row = xgb_df[xgb_df['study'] == row['study']]
    xgb_auc = xgb_row['AUC'].values[0] if len(xgb_row) else float('nan')
    print(f"{row['study']:<25} {row['AUC']:>8.4f} {xgb_auc:>9.4f}")
print("-" * 45)
print(f"{'Mean':<25} {rf_df['AUC'].mean():>8.4f} {xgb_df['AUC'].mean():>9.4f}")

In [ ]:
# ── SHAP feature importances ──

best_row    = rf_df.loc[rf_df['AUC'].idxmax()]
best_params = best_row['best_params']
print(f"Best fold : {best_row['study']}  AUC={best_row['AUC']}")
print(f"Params    : {best_params}\n")

final_rf = RandomForestClassifier(
    n_estimators    = best_params['n_estimators'],
    max_features    = best_params['max_features'],
    min_samples_leaf= best_params['min_samples_leaf'],
    class_weight    = 'balanced',
    random_state    = 42,
    n_jobs          = -1
)
final_rf.fit(X, y)
print("Final RF trained on all 520 subjects.")

print("Computing SHAP values — estimated 10–20 min...")
explainer   = shap.TreeExplainer(final_rf)
shap_values = explainer.shap_values(X)

# Use class 1 (fiber) SHAP values
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

mean_abs_shap = np.abs(sv).mean(axis=0)
shap_df = pd.DataFrame({
    'OTU_ID'        : otu_cols,
    'mean_abs_shap' : mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_df['rank'] = shap_df.index + 1

print(f"\nTop 20 OTUs by mean |SHAP|:")
print(shap_df.head(20)[['rank','OTU_ID','mean_abs_shap']].to_string(index=False))

In [ ]:
# ── Save outputs and summary ──

# Combined RF + XGBoost results
rf_df['model']  = 'RF'
xgb_df['model'] = 'XGBoost'
combined = pd.concat(
    [rf_df[['model','study','n_test','n_fiber','n_control','AUC']],
     xgb_df[['model','study','n_test','n_fiber','n_control','AUC']]],
    ignore_index=True
)
combined.to_csv(os.path.join(OUT, "delta_rf_xgb_results.csv"), index=False)
print(f"Saved : delta_rf_xgb_results.csv")

# SHAP importances — these feed directly into Analysis 2 top-200 OTU selection
shap_df.to_csv(os.path.join(OUT, "delta_feature_importances.csv"), index=False)
print(f"Saved : delta_feature_importances.csv  ({len(shap_df)} OTUs ranked)")

# ── Final summary ──────────────────────────────
print("\n" + "="*55)
print("  ANALYSIS 1 COMPLETE — DELTA CLR RF + XGBOOST")
print("="*55)
print(f"\n  RF      Mean AUC : {rf_df['AUC'].mean():.4f} ± {rf_df['AUC'].std():.4f}")
print(f"  XGBoost Mean AUC : {xgb_df['AUC'].mean():.4f} ± {xgb_df['AUC'].std():.4f}")

print(f"\n  Per-study breakdown:")
print(f"  {'Study':<25} {'RF':>7} {'XGB':>7}")
print(f"  {'-'*41}")
for _, row in rf_df.iterrows():
    xr = xgb_df[xgb_df['study'] == row['study']]['AUC'].values
    xauc = xr[0] if len(xr) else float('nan')
    print(f"  {row['study']:<25} {row['AUC']:>7.4f} {xauc:>7.4f}")

print(f"\n  Verdict:")
print(f"  Mean AUC 0.62 — study-dependent fiber-specific signal.")
print(f"  Strong in tight RCTs : Deehan (RF 0.788, XGB 0.833)")
print(f"                         Healey (RF 0.736, XGB 0.664)")
print(f"  Near-chance in hetero: Baxter, Dahl, Morales")
print(f"\n  Manuscript framing   : Delta CLR shows fiber-specific")
print(f"  microbiome change in controlled RCT designs. Overall")
print(f"  mean AUC below 0.75 threshold — Analysis 2 mixed")
print(f"  effects model is primary fiber-specificity evidence.")
print(f"\n  Next: Analysis 2 — Linear Mixed Effects Model (R)")
print("="*55)